In [0]:
# Importing libraries

import os
import datetime
import numpy as np
import pandas as pd
import mlflow
from mlflow.tracking import MlflowClient

os.environ['MLFLOW_DFS_TMP'] = '/Volumes/workspace/ml_layer/mlflow_tmp'

def ensure_catalog():
    spark.sql("USE CATALOG workspace")
    spark.sql("USE DATABASE ml_layer")

ensure_catalog()

client   = MlflowClient()
username = spark.sql("SELECT current_user()").collect()[0][0]

# Pulling experiment runs from mlflow

def fetch_runs(experiment_path):
    """Get all runs from an experiment, ordered most recent first."""
    experiment = mlflow.get_experiment_by_name(experiment_path)
    if experiment is None:
        print(f"Experiment not found: {experiment_path}")
        return pd.DataFrame()
    return mlflow.search_runs(
        experiment_ids=[experiment.experiment_id],
        order_by=["start_time DESC"]
    )


# Mapping all experiments to their model_type label
experiments = {
    f"/Users/{username}/supervised_models"     : "Supervised",
    f"/Users/{username}/anomaly_detection"     : "Anomaly",
    f"/Users/{username}/lstm"                  : "LSTM",
    f"/Users/{username}/hyperparameter_tuning" : "Optuna-tuned",
    f"/Users/{username}/combined_scoring"      : "Combined",
    f"/Users/{username}/explainability"        : "Explainability"
}

# Building unified model comparison table

def parse_model_runs(runs_df, model_type):
    """Extract one record per run with standard metric columns."""
    if runs_df.empty:
        return []

    records = []
    for _, row in runs_df.iterrows():
        run_name = row.get("tags.mlflow.runName", "")
        if pd.isna(run_name) or not run_name:
            continue

        # Skipping non-model runs
        if run_name.startswith("SHAP_") or run_name.startswith("Optuna_"):
            continue

        parts   = run_name.split("_", 1)
        model   = parts[0]
        dataset = parts[1].capitalize() if len(parts) > 1 else "Unknown"

        records.append({
            "run_name"      : run_name,
            "model"         : model,
            "dataset"       : dataset,
            "model_type"    : model_type,
            "roc_auc"       : row.get("metrics.roc_auc",        np.nan),
            "pr_auc"        : row.get("metrics.pr_auc",         np.nan),
            "f1"            : row.get("metrics.f1",             np.nan),
            "best_threshold": row.get("metrics.best_threshold", np.nan),
            "start_time"    : row.get("start_time")
        })
    return records


all_records = []
for exp_path, model_type in experiments.items():
    runs    = fetch_runs(exp_path)
    parsed  = parse_model_runs(runs, model_type)
    all_records.extend(parsed)

results_df = pd.DataFrame(all_records)

# Deduplication to keep most recent run per model and dataset
if len(results_df) > 0:
    results_df = results_df.sort_values("start_time", ascending=False) \
                           .drop_duplicates(subset=["model", "dataset"],
                                            keep="first") \
                           .reset_index(drop=True)

print(f"Loaded {len(results_df)} unique model results from MLflow")

# Round and select for dashboard
model_comparison = results_df[[
    "model", "dataset", "model_type",
    "roc_auc", "pr_auc", "f1", "best_threshold"
]].copy()

for c in ["roc_auc", "pr_auc", "f1", "best_threshold"]:
    model_comparison[c] = model_comparison[c].round(4)

spark.createDataFrame(model_comparison).write.format("delta") \
    .mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("workspace.ml_layer.dashboard_model_comparison")

# Confusion matrix from supervised_models MLflow runs

def parse_confusion_matrix(runs_df):
    records = []
    for _, row in runs_df.iterrows():
        run_name = row.get("tags.mlflow.runName", "")
        if pd.isna(run_name) or not run_name:
            continue

        tp = row.get("metrics.true_positives",  np.nan)
        fp = row.get("metrics.false_positives", np.nan)
        fn = row.get("metrics.false_negatives", np.nan)
        tn = row.get("metrics.true_negatives",  np.nan)

        # Only include runs that logged confusion matrix
        if all(pd.notna(v) for v in [tp, fp, fn, tn]):
            parts   = run_name.split("_", 1)
            model   = parts[0]
            dataset = parts[1].capitalize() if len(parts) > 1 else "Unknown"

            records.append({
                "model"           : model,
                "dataset"         : dataset,
                "true_positives"  : int(tp),
                "false_positives" : int(fp),
                "false_negatives" : int(fn),
                "true_negatives"  : int(tn),
                "fraud_caught"    : int(tp),
                "fraud_missed"    : int(fn),
                "false_alerts"    : int(fp),
                "precision"       : round(tp / (tp + fp), 4) if (tp + fp) > 0 else 0,
                "recall"          : round(tp / (tp + fn), 4) if (tp + fn) > 0 else 0,
                "start_time"      : row.get("start_time")
            })
    return records


supervised_runs   = fetch_runs(f"/Users/{username}/supervised_models")
confusion_records = parse_confusion_matrix(supervised_runs)

if confusion_records:
    confusion_df = pd.DataFrame(confusion_records) \
                     .sort_values("start_time", ascending=False) \
                     .drop_duplicates(subset=["model", "dataset"], keep="first") \
                     .drop(columns=["start_time"])
    
    spark.createDataFrame(confusion_df).write.format("delta") \
        .mode("overwrite").option("overwriteSchema", "true") \
        .saveAsTable("workspace.ml_layer.dashboard_confusion_matrix")
    print(f"Saved confusion matrix for {len(confusion_df)} models")
else:
    print("⚠️  No confusion matrix metrics found in MLflow")
    print("   Make sure notebook 01 logs TP/FP/FN/TN with mlflow.log_metric()")

# Feature importance from MLflow SHAP artifacts

def parse_feature_importance():
    exp_path = f"/Users/{username}/explainability"
    runs     = fetch_runs(exp_path)

    if runs.empty:
        return []

    # Filter for SHAP runs
    shap_runs = runs[runs["tags.mlflow.runName"].str.startswith("SHAP_", na=False)]

    records = []
    for _, row in shap_runs.iterrows():
        run_id   = row.get("run_id")
        run_name = row.get("tags.mlflow.runName", "")
        dataset  = run_name.replace("SHAP_", "").capitalize()

        try:
            artifact_path = client.download_artifacts(
                run_id, f"shap_importance_{run_name.replace('SHAP_', '')}.json"
            )
            with open(artifact_path) as f:
                import json
                shap_data = json.load(f)

            for item in shap_data:
                records.append({
                    "feature"   : item["feature"],
                    "dataset"   : dataset,
                    "importance": round(float(item["mean_abs_shap"]), 4),
                    "rank"      : int(item["rank"]),
                    "run_id"    : run_id
                })
        except Exception as e:
            print(f"  Could not load SHAP artifact for {run_name}: {e}")

    return records


shap_records = parse_feature_importance()

if shap_records:
    shap_df = pd.DataFrame(shap_records) \
                .drop_duplicates(subset=["feature", "dataset"], keep="first") \
                .drop(columns=["run_id"])
    
    spark.createDataFrame(shap_df).write.format("delta") \
        .mode("overwrite").option("overwriteSchema", "true") \
        .saveAsTable("workspace.ml_layer.dashboard_feature_importance")
    print(f"Saved feature importance: {len(shap_df)} feature-dataset pairs")
else:
    print("⚠️  No SHAP artifacts found in explainability experiment")

# Fraud rate summary from feature tables

from pyspark.sql.functions import col

def fraud_summary(table_name, dataset_label, split_label):
    df          = spark.table(table_name)
    total       = df.count()
    fraud       = df.filter(col("is_fraud") == 1).count()

    return {
        "dataset"        : dataset_label,
        "split"          : split_label,
        "total_records"  : total,
        "fraud_records"  : fraud,
        "legit_records"  : total - fraud,
        "fraud_rate_pct" : round(100 * fraud / total, 2)
    }


fraud_records = [
    fraud_summary("workspace.ml_layer.application_train_features", "Applications", "Train"),
    fraud_summary("workspace.ml_layer.application_test_features",  "Applications", "Test"),
    fraud_summary("workspace.ml_layer.transaction_train_features", "Transactions", "Train"),
    fraud_summary("workspace.ml_layer.transaction_test_features",  "Transactions", "Test"),
]

spark.createDataFrame(pd.DataFrame(fraud_records)).write.format("delta") \
    .mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("workspace.ml_layer.dashboard_fraud_summary")

print(f"Computed fraud summary directly from feature tables")

# Refreshing metadata

refresh_metadata = pd.DataFrame([{
    "last_refresh"        : datetime.datetime.now().isoformat(),
    "models_loaded"       : len(model_comparison),
    "experiments_scanned" : len(experiments),
    "data_source"         : "MLflow"
}])

spark.createDataFrame(refresh_metadata).write.format("delta") \
    .mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("workspace.ml_layer.dashboard_refresh_metadata")

# SUMMARY

print("\n" + "="*60)
print("  DASHBOARD DATA REFRESH COMPLETE")
print("="*60)
print(f"  Source             : MLflow experiments")
print(f"  Tables refreshed   : 5")
print(f"  Models loaded      : {len(model_comparison)}")
print(f"  Refresh timestamp  : {datetime.datetime.now()}")